In [1]:
import numpy as np
import cv2
from scipy import ndimage
import pandas as pd
from datetime import datetime
import os
from glob import glob

class MultiClassMaskAnalyzer:
    def __init__(self, num_classes=18):
        self.num_classes = num_classes
    
    def load_mask(self, mask_path):
        """Load 1-channel mask with class IDs 0-17"""
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Could not load mask from {mask_path}")
        
        # Validate that it's a proper multi-class mask
        unique_vals = np.unique(mask)
        if not all(0 <= val < self.num_classes for val in unique_vals):
            print(f"Warning: Mask {mask_path} has values outside 0-{self.num_classes-1}: {unique_vals}")
        
        return mask.astype(np.uint8)
    
    def analyze_multi_class_mask(self, mask):
        """Analyze the entire multi-class mask as a whole"""
        
        # 1. Class Distribution Analysis
        class_counts = []
        for class_id in range(self.num_classes):
            count = np.sum(mask == class_id)
            class_counts.append(count)
        
        total_pixels = mask.shape[0] * mask.shape[1]
        class_distribution = [count / total_pixels for count in class_counts]
        
        # 2. Boundary Quality (across all class boundaries)
        boundary_score = self.calculate_boundary_quality(mask)
        
        # 3. Region Coherence
        coherence_score = self.calculate_region_coherence(mask)
        
        # 4. Segmentation Cleanliness
        cleanliness_score = self.calculate_cleanliness(mask)
        
        # Overall quality score
        overall_quality = 0.4 * boundary_score + 0.3 * coherence_score + 0.3 * cleanliness_score
        
        return {
            'boundary_score': boundary_score,
            'coherence_score': coherence_score,
            'cleanliness_score': cleanliness_score,
            'overall_quality': overall_quality,
            'classes_present': len([c for c in class_counts if c > 0]),
            'class_distribution': class_distribution
        }
    
    def calculate_boundary_quality(self, mask):
        """Calculate smoothness of boundaries between different classes"""
        # Create a boundary map where different classes meet
        boundary_map = np.zeros_like(mask, dtype=bool)
        
        # Check 4-connected neighbors for class transitions
        h, w = mask.shape
        for i in range(1, h-1):
            for j in range(1, w-1):
                current_class = mask[i, j]
                # Check if any neighbor has different class
                if (mask[i-1, j] != current_class or 
                    mask[i+1, j] != current_class or 
                    mask[i, j-1] != current_class or 
                    mask[i, j+1] != current_class):
                    boundary_map[i, j] = True
        
        # Calculate boundary complexity
        boundary_pixels = np.sum(boundary_map)
        if boundary_pixels == 0:
            return 0.0
        
        # Simple boundary score - fewer fragmented boundaries is better
        labeled, num_boundary_regions = ndimage.label(boundary_map)
        fragmentation = num_boundary_regions / boundary_pixels if boundary_pixels > 0 else 0
        
        return 1.0 / (1.0 + fragmentation * 10)
    
    def calculate_region_coherence(self, mask):
        """Check if regions of same class are coherent/connected"""
        total_coherence = 0
        classes_checked = 0
        
        for class_id in range(self.num_classes):
            class_mask = (mask == class_id).astype(np.uint8)
            if np.sum(class_mask) > 0:  # If class is present
                labeled, num_regions = ndimage.label(class_mask)
                if num_regions > 0:
                    region_sizes = [np.sum(labeled == i) for i in range(1, num_regions + 1)]
                    largest_region = max(region_sizes)
                    total_pixels = np.sum(class_mask)
                    
                    # Coherence: how concentrated is this class?
                    coherence = largest_region / total_pixels if total_pixels > 0 else 0
                    total_coherence += coherence
                    classes_checked += 1
        
        return total_coherence / classes_checked if classes_checked > 0 else 0
    
    def calculate_cleanliness(self, mask):
        """Check for small isolated regions that might be noise"""
        total_cleanliness = 0
        classes_checked = 0
        
        for class_id in range(1, self.num_classes):  # Skip background (class 0)
            class_mask = (mask == class_id).astype(np.uint8)
            if np.sum(class_mask) > 0:
                labeled, num_regions = ndimage.label(class_mask)
                if num_regions > 0:
                    region_sizes = [np.sum(labeled == i) for i in range(1, num_regions + 1)]
                    total_pixels = np.sum(class_mask)
                    
                    # Penalty for too many small regions
                    small_regions = sum(1 for size in region_sizes if size < total_pixels * 0.01)
                    cleanliness = 1.0 - min(small_regions * 0.1, 0.5)
                    total_cleanliness += cleanliness
                    classes_checked += 1
        
        return total_cleanliness / classes_checked if classes_checked > 0 else 1.0
    
    def get_quality_category(self, confidence):
        if confidence >= 0.8:
            return "Good"
        elif confidence >= 0.5:
            return "Fair"
        else:
            return "Bad"
    
    def analyze_masks_from_directory(self, masks_dir, mask_extensions=['.png']):
        """Analyze all multi-class masks from directory"""
        mask_paths = []
        for ext in mask_extensions:
            mask_paths.extend(glob(os.path.join(masks_dir, f"*{ext}")))
        
        print(f"Found {len(mask_paths)} mask files in {masks_dir}")
        
        results = []
        
        for i, mask_path in enumerate(mask_paths):
            if i % 50 == 0:
                print(f"Processed {i}/{len(mask_paths)} masks...")
            
            try:
                mask = self.load_mask(mask_path)
                analysis = self.analyze_multi_class_mask(mask)
                
                result = {
                    'mask_path': mask_path,
                    'mask_name': os.path.basename(mask_path),
                    'overall_quality': round(analysis['overall_quality'], 4),
                    'quality_category': self.get_quality_category(analysis['overall_quality']),
                    'boundary_score': round(analysis['boundary_score'], 4),
                    'coherence_score': round(analysis['coherence_score'], 4),
                    'cleanliness_score': round(analysis['cleanliness_score'], 4),
                    'classes_present': analysis['classes_present'],
                }
                
                # Add class distribution for top classes
                for class_id, dist in enumerate(analysis['class_distribution']):
                    if dist > 0.01:  # Only significant classes
                        result[f'class_{class_id}_pct'] = round(dist * 100, 2)
                
                results.append(result)
                
            except Exception as e:
                print(f"Error processing {mask_path}: {e}")
                results.append({
                    'mask_path': mask_path,
                    'mask_name': os.path.basename(mask_path),
                    'overall_quality': 0.0,
                    'quality_category': 'Bad',
                    'boundary_score': 0.0,
                    'coherence_score': 0.0,
                    'cleanliness_score': 0.0,
                    'classes_present': 0,
                })
        
        return results
    
    def generate_comprehensive_report(self, results):
        """Generate detailed report"""
        df = pd.DataFrame(results)
        
        total_masks = len(df)
        good_count = len(df[df['quality_category'] == 'Good'])
        fair_count = len(df[df['quality_category'] == 'Fair'])
        bad_count = len(df[df['quality_category'] == 'Bad'])
        
        report = f"""
=== MULTI-CLASS MASK QUALITY REPORT ===
Total Masks: {total_masks}
Good: {good_count} ({good_count/total_masks*100:.1f}%)
Fair: {fair_count} ({fair_count/total_masks*100:.1f}%)
Bad: {bad_count} ({bad_count/total_masks*100:.1f}%)

Average Quality: {df['overall_quality'].mean():.3f}
Average Classes per Mask: {df['classes_present'].mean():.1f}

--- WORST 10 MASKS ---
"""
        worst_masks = df.nsmallest(10, 'overall_quality')
        for _, row in worst_masks.iterrows():
            report += f"{row['mask_name']}: {row['overall_quality']:.3f} ({row['quality_category']})\n"
        
        return report, df

# Usage
def main():
    analyzer = MultiClassMaskAnalyzer(num_classes=18)
    MASKS_DIR = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_finetune_sam_output_pointprompts"
    
    results = analyzer.analyze_masks_from_directory(MASKS_DIR)
    report, df = analyzer.generate_comprehensive_report(results)
    print(report)
    
    df.to_csv("multiclass_quality_report.csv", index=False)
    print("Report saved to multiclass_quality_report.csv")
    
    return results

if __name__ == "__main__":
    main()

Found 2100 mask files in /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_finetune_sam_output_pointprompts
Processed 0/2100 masks...
Processed 50/2100 masks...
Processed 100/2100 masks...
Processed 150/2100 masks...
Processed 200/2100 masks...
Processed 250/2100 masks...
Processed 300/2100 masks...
Processed 350/2100 masks...
Processed 400/2100 masks...
Processed 450/2100 masks...
Processed 500/2100 masks...
Processed 550/2100 masks...
Processed 600/2100 masks...
Processed 650/2100 masks...
Processed 700/2100 masks...
Processed 750/2100 masks...
Processed 800/2100 masks...
Processed 850/2100 masks...
Processed 900/2100 masks...
Processed 950/2100 masks...
Processed 1000/2100 masks...
Processed 1050/2100 masks...
Processed 1100/2100 masks...
Processed 1150/2100 masks...
Processed 1200/2100 masks...
Processed 1250/2100 masks...
Processed 1300/2100 masks...
Processed 1350/2100 masks...
Processed 1400/2100 masks...
Processed 1450/2100 masks...
Processed 1500/2100 masks...
Processed 

In [5]:
import numpy as np
import cv2
from scipy import ndimage
import pandas as pd
from datetime import datetime
import os
from glob import glob

class MaskQualityAnalyzer:
    def __init__(self, num_classes=18):
        self.num_classes = num_classes
    
    def load_mask(self, mask_path):
        """Load 1-channel mask with class IDs 0-17"""
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Could not load mask from {mask_path}")
        return mask.astype(np.uint8)
    
    def calculate_iou_per_class(self, pred_mask, gt_mask):
        """Calculate IoU for each class"""
        ious = []
        for class_id in range(self.num_classes):
            pred_class = (pred_mask == class_id)
            gt_class = (gt_mask == class_id)
            
            intersection = np.logical_and(pred_class, gt_class).sum()
            union = np.logical_or(pred_class, gt_class).sum()
            
            if union == 0:
                iou = 1.0  # Both are empty
            else:
                iou = intersection / union
            ious.append(iou)
        
        return ious
    
    def calculate_miou(self, pred_mask, gt_mask):
        """Calculate mean IoU across all classes"""
        ious = self.calculate_iou_per_class(pred_mask, gt_mask)
        # Average over classes that are present in GT
        present_classes = [iou for class_id, iou in enumerate(ious) if np.sum(gt_mask == class_id) > 0]
        if len(present_classes) == 0:
            return 0.0
        return np.mean(present_classes)
    
    def analyze_multi_class_mask(self, mask):
        """Analyze the quality of multi-class mask (no GT needed)"""
        # Class Distribution Analysis
        class_counts = []
        for class_id in range(self.num_classes):
            count = np.sum(mask == class_id)
            class_counts.append(count)
        
        total_pixels = mask.shape[0] * mask.shape[1]
        
        # Boundary Quality
        boundary_score = self.calculate_boundary_quality(mask)
        
        # Region Coherence
        coherence_score = self.calculate_region_coherence(mask)
        
        # Segmentation Cleanliness
        cleanliness_score = self.calculate_cleanliness(mask)
        
        # Overall quality score
        overall_quality = 0.4 * boundary_score + 0.3 * coherence_score + 0.3 * cleanliness_score
        
        return {
            'boundary_score': boundary_score,
            'coherence_score': coherence_score,
            'cleanliness_score': cleanliness_score,
            'overall_quality': overall_quality,
            'classes_present': len([c for c in class_counts if c > 0]),
        }
    
    def calculate_boundary_quality(self, mask):
        """Calculate smoothness of boundaries between different classes"""
        boundary_map = np.zeros_like(mask, dtype=bool)
        h, w = mask.shape
        
        for i in range(1, h-1):
            for j in range(1, w-1):
                current_class = mask[i, j]
                if (mask[i-1, j] != current_class or mask[i+1, j] != current_class or 
                    mask[i, j-1] != current_class or mask[i, j+1] != current_class):
                    boundary_map[i, j] = True
        
        boundary_pixels = np.sum(boundary_map)
        if boundary_pixels == 0:
            return 0.0
        
        labeled, num_boundary_regions = ndimage.label(boundary_map)
        fragmentation = num_boundary_regions / boundary_pixels if boundary_pixels > 0 else 0
        return 1.0 / (1.0 + fragmentation * 10)
    
    def calculate_region_coherence(self, mask):
        """Check if regions of same class are coherent/connected"""
        total_coherence = 0
        classes_checked = 0
        
        for class_id in range(self.num_classes):
            class_mask = (mask == class_id).astype(np.uint8)
            if np.sum(class_mask) > 0:
                labeled, num_regions = ndimage.label(class_mask)
                if num_regions > 0:
                    region_sizes = [np.sum(labeled == i) for i in range(1, num_regions + 1)]
                    largest_region = max(region_sizes)
                    total_pixels = np.sum(class_mask)
                    coherence = largest_region / total_pixels if total_pixels > 0 else 0
                    total_coherence += coherence
                    classes_checked += 1
        
        return total_coherence / classes_checked if classes_checked > 0 else 0
    
    def calculate_cleanliness(self, mask):
        """Check for small isolated regions that might be noise"""
        total_cleanliness = 0
        classes_checked = 0
        
        for class_id in range(1, self.num_classes):  # Skip background
            class_mask = (mask == class_id).astype(np.uint8)
            if np.sum(class_mask) > 0:
                labeled, num_regions = ndimage.label(class_mask)
                if num_regions > 0:
                    region_sizes = [np.sum(labeled == i) for i in range(1, num_regions + 1)]
                    total_pixels = np.sum(class_mask)
                    small_regions = sum(1 for size in region_sizes if size < total_pixels * 0.01)
                    cleanliness = 1.0 - min(small_regions * 0.1, 0.5)
                    total_cleanliness += cleanliness
                    classes_checked += 1
        
        return total_cleanliness / classes_checked if classes_checked > 0 else 1.0
    
    def get_quality_category(self, confidence):
        if confidence >= 0.7:
            return "Good"
        elif confidence >= 0.5:
            return "Fair"
        else:
            return "Bad"
    
    def find_corresponding_gt(self, pred_mask_path, gt_dir):
        """Find corresponding ground truth mask - handles 'mask_' prefix difference"""
        pred_name = os.path.basename(pred_mask_path)
        
        # Remove 'mask_' prefix from predicted mask name to get GT name
        if pred_name.startswith('mask_'):
            gt_name = pred_name[5:]  # Remove 'mask_' prefix (5 characters)
        else:
            gt_name = pred_name
        
        # Try the name without 'mask_' prefix
        gt_path = os.path.join(gt_dir, gt_name)
        if os.path.exists(gt_path):
            return gt_path
        
        # Also try with different extensions
        base_name = os.path.splitext(gt_name)[0]
        for ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']:
            gt_path = os.path.join(gt_dir, base_name + ext)
            if os.path.exists(gt_path):
                return gt_path
        
        # If still not found, try other common patterns
        possible_names = [
            gt_name,
            gt_name.replace('pred', 'gt'),
            gt_name.replace('predicted', 'ground_truth'),
            gt_name.replace('_pred', '_gt'),
            base_name + '_gt.png',
            base_name + '_mask.png',
        ]
        
        for name in possible_names:
            gt_path = os.path.join(gt_dir, name)
            if os.path.exists(gt_path):
                return gt_path
        
        raise FileNotFoundError(f"Could not find GT for {pred_name}. Tried: {gt_name} and variations")
    
    def analyze_all_masks_with_gt(self, pred_masks_dir, gt_masks_dir):
        """Analyze all predicted masks with their ground truth"""
        pred_mask_paths = glob(os.path.join(pred_masks_dir, "*.png"))
        
        #print(f"Found {len(pred_mask_paths)} predicted mask files")
        #print(f"Predicted masks have names like: {os.path.basename(pred_mask_paths[0]) if pred_mask_paths else 'No masks found'}")
        
        results = []
        
        for i, pred_path in enumerate(pred_mask_paths):
            if i % 50 == 0:
                print(f"Processed {i}/{len(pred_mask_paths)} masks...")
            
            try:
                # Load predicted mask
                pred_mask = self.load_mask(pred_path)
                
                # Find and load corresponding GT
                gt_path = self.find_corresponding_gt(pred_path, gt_masks_dir)
                gt_mask = self.load_mask(gt_path)
                
                #print(f"Found GT: {os.path.basename(gt_path)} for {os.path.basename(pred_path)}")
                
                # Calculate mIoU
                miou = self.calculate_miou(pred_mask, gt_mask)
                
                # Calculate quality metrics (no GT needed)
                quality_analysis = self.analyze_multi_class_mask(pred_mask)
                
                result = {
                    'mask_name': os.path.basename(pred_path),
                    'gt_name': os.path.basename(gt_path),
                    'miou_score': round(miou, 4),
                    'quality_score': round(quality_analysis['overall_quality'], 4),
                    'quality_category': self.get_quality_category(quality_analysis['overall_quality']),
                    'boundary_score': round(quality_analysis['boundary_score'], 4),
                    'coherence_score': round(quality_analysis['coherence_score'], 4),
                    'cleanliness_score': round(quality_analysis['cleanliness_score'], 4),
                    'classes_present': quality_analysis['classes_present'],
                    'gt_found': True
                }
                
                results.append(result)
                
            except FileNotFoundError as e:
                print(f"GT not found for {os.path.basename(pred_path)}: {e}")
                # Analyze quality only (without mIoU)
                pred_mask = self.load_mask(pred_path)
                quality_analysis = self.analyze_multi_class_mask(pred_mask)
                
                result = {
                    'mask_name': os.path.basename(pred_path),
                    'gt_name': 'NOT_FOUND',
                    'miou_score': 0.0,
                    'quality_score': round(quality_analysis['overall_quality'], 4),
                    'quality_category': self.get_quality_category(quality_analysis['overall_quality']),
                    'boundary_score': round(quality_analysis['boundary_score'], 4),
                    'coherence_score': round(quality_analysis['coherence_score'], 4),
                    'cleanliness_score': round(quality_analysis['cleanliness_score'], 4),
                    'classes_present': quality_analysis['classes_present'],
                    'gt_found': False
                }
                results.append(result)
                
            except Exception as e:
                print(f"Error processing {pred_path}: {e}")
                results.append({
                    'mask_name': os.path.basename(pred_path),
                    'gt_name': 'ERROR',
                    'miou_score': 0.0,
                    'quality_score': 0.0,
                    'quality_category': 'Bad',
                    'boundary_score': 0.0,
                    'coherence_score': 0.0,
                    'cleanliness_score': 0.0,
                    'classes_present': 0,
                    'gt_found': False
                })
        
        return results
    
    def generate_comprehensive_report(self, results):
        """Generate detailed report with mIoU and quality analysis"""
        df = pd.DataFrame(results)
        
        # Filter masks where GT was found
        df_with_gt = df[df['gt_found'] == True]
        
        total_masks = len(df)
        masks_with_gt = len(df_with_gt)
        
        # Quality distribution
        good_count = len(df[df['quality_category'] == 'Good'])
        fair_count = len(df[df['quality_category'] == 'Fair'])
        bad_count = len(df[df['quality_category'] == 'Bad'])
        
        # Average scores
        avg_miou = df_with_gt['miou_score'].mean() if masks_with_gt > 0 else 0
        avg_quality = df['quality_score'].mean()
        
        report = f"""
=== COMPREHENSIVE MASK ANALYSIS REPORT ===
Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Total Masks Analyzed: {total_masks}
Masks with Ground Truth: {masks_with_gt}

--- QUALITY DISTRIBUTION ---
Good Masks:  {good_count} ({good_count/total_masks*100:.1f}%)
Fair Masks:  {fair_count} ({fair_count/total_masks*100:.1f}%)
Bad Masks:   {bad_count} ({bad_count/total_masks*100:.1f}%)

--- AVERAGE SCORES ---
Average mIoU (with GT): {avg_miou:.4f}
Average Quality Score:  {avg_quality:.4f}

--- CORRELATION ANALYSIS ---
"""
        if masks_with_gt > 0:
            correlation = df_with_gt['miou_score'].corr(df_with_gt['quality_score'])
            report += f"Correlation (mIoU vs Quality): {correlation:.4f}\n"
        
        # Worst 10 masks by mIoU (if available) or quality
        if masks_with_gt > 0:
            worst_masks = df_with_gt.nsmallest(10, 'miou_score')[['mask_name', 'miou_score', 'quality_score', 'quality_category']]
            report += "\nTOP 10 WORST MASKS (by mIoU):\n"
            for _, row in worst_masks.iterrows():
                report += f"  {row['mask_name']}: mIoU={row['miou_score']:.4f}, Quality={row['quality_score']:.3f} ({row['quality_category']})\n"
        else:
            worst_masks = df.nsmallest(10, 'quality_score')[['mask_name', 'quality_score', 'quality_category']]
            report += "\nTOP 10 WORST MASKS (by Quality):\n"
            for _, row in worst_masks.iterrows():
                report += f"  {row['mask_name']}: Quality={row['quality_score']:.3f} ({row['quality_category']})\n"
        
        return report, df

# Main function
def main():
    analyzer = MaskQualityAnalyzer(num_classes=18)
    
    # === CONFIGURE YOUR PATHS HERE ===
    PRED_MASKS_DIR = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_finetune_sam_output_pointprompts"  # Contains files like "mask_agriculture.png"
    GT_MASKS_DIR = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_1cmasks"  # Contains files like "agriculture.png" (no "mask_" prefix)
    
    if not os.path.exists(PRED_MASKS_DIR):
        print(f"Error: Predicted masks directory not found: {PRED_MASKS_DIR}")
        return
    
    if not os.path.exists(GT_MASKS_DIR):
        print(f"Error: Ground truth directory not found: {GT_MASKS_DIR}")
        return
    
    # Analyze all masks with GT
    print("Starting comprehensive analysis with mIoU...")
    results = analyzer.analyze_all_masks_with_gt(PRED_MASKS_DIR, GT_MASKS_DIR)
    
    # Generate report
    print("\nGenerating report...")
    report, df = analyzer.generate_comprehensive_report(results)
    print(report)
    
    # Save detailed results to CSV
    output_file = "comprehensive_mask_analysis.csv"
    df.to_csv(output_file, index=False)
    print(f"\nDetailed results saved to: {output_file}")
    
    # Summary
    masks_with_gt = len([r for r in results if r['gt_found']])
    good_masks = len([r for r in results if r['quality_category'] == 'Good'])
    
    print(f"\n=== QUICK SUMMARY ===")
    print(f"Total masks: {len(results)}")
    print(f"Masks with GT: {masks_with_gt}")
    print(f"Good quality masks: {good_masks}")
    
    if masks_with_gt > 0:
        avg_miou = np.mean([r['miou_score'] for r in results if r['gt_found']])
        print(f"Average mIoU: {avg_miou:.4f}")
    
    return results

if __name__ == "__main__":
    results = main()

Starting comprehensive analysis with mIoU...
Processed 0/2100 masks...
Processed 50/2100 masks...
Processed 100/2100 masks...
Processed 150/2100 masks...
Processed 200/2100 masks...
Processed 250/2100 masks...
Processed 300/2100 masks...
Processed 350/2100 masks...
Processed 400/2100 masks...
Processed 450/2100 masks...
Processed 500/2100 masks...
Processed 550/2100 masks...
Processed 600/2100 masks...
Processed 650/2100 masks...
Processed 700/2100 masks...
Processed 750/2100 masks...
Processed 800/2100 masks...
Processed 850/2100 masks...
Processed 900/2100 masks...
Processed 950/2100 masks...
Processed 1000/2100 masks...
Processed 1050/2100 masks...
Processed 1100/2100 masks...
Processed 1150/2100 masks...
Processed 1200/2100 masks...
Processed 1250/2100 masks...
Processed 1300/2100 masks...
Processed 1350/2100 masks...
Processed 1400/2100 masks...
Processed 1450/2100 masks...
Processed 1500/2100 masks...
Processed 1550/2100 masks...
Processed 1600/2100 masks...
Processed 1650/2100 m

In [ ]:
PRED_MASKS_DIR = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_finetune_sam_output_pointprompts" # Your refined SAM predictions
GT_MASKS_DIR = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_1cmasks"  # Ground truth masks
    